# 04 Merge Datasets

Merge cleaned Egyptian and USDA datasets into the MVP `foods_master` dataset.

Rules: Egyptian foods have priority. USDA fills missing foods only. No aggressive merging.

Output: `data/processed/foods_master.parquet`

In [ ]:
from pathlib import Path
import re

import numpy as np
import pandas as pd

ROOT = Path.cwd()
OUTPUT_DIR = ROOT / 'data' / 'processed'
EGYPTIAN_PATH = OUTPUT_DIR / 'egyptian_food_clean.parquet'
USDA_PATH = OUTPUT_DIR / 'usda_food_clean.parquet'
OUTPUT_PATH = OUTPUT_DIR / 'foods_master.parquet'

FINAL_COLUMNS = [
    'food_id', 'food_name_en', 'food_name_ar', 'food_group', 'meal_types',
    'serving_name', 'serving_weight_g', 'nutrition_basis', 'calories',
    'protein', 'carbs', 'fat', 'fiber', 'diet_tags', 'allergens',
    'is_composite_dish', 'source'
]

def require_file(path: Path) -> None:
    if not path.exists():
        raise FileNotFoundError(f'Missing required input: {path}. Run earlier notebooks first.')

def name_key(value: object) -> str:
    if pd.isna(value):
        return ''
    text = str(value).lower().strip()
    text = re.sub(r'[^a-z0-9]+', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

def enforce_schema(df: pd.DataFrame) -> pd.DataFrame:
    output = df.copy()
    for col in FINAL_COLUMNS:
        if col not in output.columns:
            output[col] = ''
    output = output[FINAL_COLUMNS].copy()
    text_cols = ['food_id', 'food_name_en', 'food_name_ar', 'food_group', 'meal_types', 'serving_name', 'nutrition_basis', 'diet_tags', 'allergens', 'source']
    for col in text_cols:
        output[col] = output[col].fillna('').astype(str)
    number_cols = ['serving_weight_g', 'calories', 'protein', 'carbs', 'fat', 'fiber']
    for col in number_cols:
        output[col] = pd.to_numeric(output[col], errors='coerce').fillna(0).astype(float)
    output['is_composite_dish'] = output['is_composite_dish'].fillna(False).astype(bool)
    return output

require_file(EGYPTIAN_PATH)
require_file(USDA_PATH)
egyptian = enforce_schema(pd.read_parquet(EGYPTIAN_PATH, engine='pyarrow'))
usda = enforce_schema(pd.read_parquet(USDA_PATH, engine='pyarrow'))

egyptian['merge_key'] = egyptian['food_name_en'].map(name_key)
usda['merge_key'] = usda['food_name_en'].map(name_key)

egyptian = egyptian.sort_values('food_name_en').drop_duplicates('merge_key', keep='first')
egyptian_keys = set(egyptian['merge_key'])
usda_fill = usda[~usda['merge_key'].isin(egyptian_keys)].copy()
usda_fill = usda_fill.sort_values('food_name_en').drop_duplicates('merge_key', keep='first')

foods_master = pd.concat([egyptian, usda_fill], ignore_index=True)
foods_master = foods_master.drop(columns=['merge_key'])
foods_master = enforce_schema(foods_master)
foods_master = foods_master.sort_values(['source', 'food_name_en']).reset_index(drop=True)
foods_master.to_parquet(OUTPUT_PATH, index=False, engine='pyarrow')

print(f'Egyptian rows kept: {len(egyptian)}')
print(f'USDA fill rows kept: {len(usda_fill)}')
print(f'foods_master rows exported: {len(foods_master)}')
print(f'Output: {OUTPUT_PATH}')
print(foods_master.head(10))
